# Lesson 09 Lab — NoC Routing, Buffers, and Contention

**Puzzle:** Why can on-chip data movement slow down even when every link is individually fast?

This notebook retains one complete RTX 5090 execution.


## Why this matters

A Network-on-Chip connects SMs, L2 slices, controllers, and other units through routers and links. Packets are split into flits; routers buffer inputs, choose routes, arbitrate virtual channels and switches, then traverse crossbars and physical links. When several flows request the same output, the link capacity is shared and queues grow. Backpressure carries the consequence upstream.


## 0. Predict before running

1. Predict which traffic pattern builds the longer queue.
2. Explain why more buffering changes burst tolerance but not steady service rate.
3. Name the hardware counters needed for a native claim.

For each prediction, write the observation that would disprove it.


## 1. Theory and mechanism

The notebook runs a deterministic discrete-time queue model. A balanced pattern spreads sources across destinations; a hotspot pattern targets one output. Offered traffic and per-link service stay fixed. Queue area, maximum queue, delivered flits, and latency expose the congestion mechanism. The model does not claim NVIDIA's private topology, router width, or arbitration policy.

- Bandwidth is a property of a path and traffic pattern, not only one link.
- Buffers absorb bursts but cannot fix sustained oversubscription.
- Backpressure turns a local hotspot into upstream stalls.


## 2. Trace the mechanism

### Mechanism map

```mermaid
flowchart LR
  A["input flits"] --> B["input buffers / VCs"]
  B --> C["route + switch arbitration"]
  C --> D["crossbar"]
  D --> E["physical link"]
  E --> F["downstream router"]
  F -->|"credits/backpressure"| B
```


## 3. Inspect the visual boundary

![Conceptual NoC router and links](../assets/NoC_on_chip_network_circuit_structure.png)

- [Printable NoC diagram](../assets/NoC_on_chip_network_circuit_structure_A4_portrait.pdf)

These are conceptual teaching diagrams. They explain the named data path and are not die-accurate schematics of a particular commercial GPU.


## 4. Inspect the execution environment

The next cell asserts CUDA, records GPU/PyTorch/CUDA identity, fixes the seed, and defines the common event-timing helpers.


In [1]:
LESSON_NO = 9
LESSON_TITLE = 'NoC Routing, Buffers, and Contention'

from pathlib import Path
from collections import Counter, deque
import json, math, platform, statistics, sys, time

import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "Chapter 04 retained runs require a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260813 + LESSON_NO
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

major, minor = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
ENV = {
    "gpu": torch.cuda.get_device_name(0),
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    pos = (len(ordered) - 1) * q
    lo, hi = math.floor(pos), math.ceil(pos)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def cuda_samples(fn, warmup=5, repeats=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        stop = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        stop.record()
        stop.synchronize()
        samples.append(float(start.elapsed_time(stop)))
    return samples

def summary(samples):
    return {
        "median_ms": statistics.median(samples),
        "p95_ms": percentile(samples, 0.95),
        "samples_ms": samples,
    }


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "seed": 20260822
}


## 5. Freeze the experiment

| Role | Frozen value |
|---|---|
| Baseline | four sources distributed across four outputs |
| Candidate | the same sources targeting one hotspot output |
| Held constant | arrival schedule, output service rate, ticks, and queue discipline |
| Measurements | delivered flits, mean/max queue, and mean latency |
| Evidence | `numerical-model` |

**Experiment:** Compare balanced and hotspot traffic in a bounded queue simulator.


## 6. Inspect the code

The simulator records enqueue time for each flit, services one flit per output per tick, and drains after arrivals stop. Identical demand makes destination concentration the independent variable.

Do not run until the code matches the frozen table.


In [2]:
def simulate(hotspot, arrival_ticks=160, sources=4, outputs=4, service_per_output=1):
    queues = [deque() for _ in range(outputs)]
    latencies = []
    max_queue = 0
    queue_area = 0
    delivered = 0
    tick = 0
    while tick < arrival_ticks or any(queues):
        if tick < arrival_ticks:
            for source in range(sources):
                destination = 0 if hotspot else source % outputs
                queues[destination].append((source, tick))
        for output in range(outputs):
            for _ in range(service_per_output):
                if queues[output]:
                    _, created = queues[output].popleft()
                    latencies.append(tick - created + 1)
                    delivered += 1
        total_queued = sum(len(q) for q in queues)
        queue_area += total_queued
        max_queue = max(max_queue, max(len(q) for q in queues))
        tick += 1
    return {
        "delivered_flits": delivered, "drain_ticks": tick,
        "mean_latency_ticks": statistics.mean(latencies),
        "p95_latency_ticks": percentile(latencies, 0.95),
        "mean_queue": queue_area / tick, "max_queue": max_queue,
    }

balanced = simulate(False)
hotspot = simulate(True)
metrics = {
    "balanced": balanced, "hotspot": hotspot,
    "hotspot_latency_ratio": hotspot["mean_latency_ticks"] / balanced["mean_latency_ticks"],
}
analysis = (
    f"Balanced and hotspot traffic delivered the same {balanced['delivered_flits']} flits, "
    f"but mean latency changed from {balanced['mean_latency_ticks']:.2f} to "
    f"{hotspot['mean_latency_ticks']:.2f} ticks as one output became oversubscribed."
)
print(json.dumps(metrics, indent=2))


{
  "balanced": {
    "delivered_flits": 640,
    "drain_ticks": 160,
    "mean_latency_ticks": 1,
    "p95_latency_ticks": 1.0,
    "mean_queue": 0.0,
    "max_queue": 0
  },
  "hotspot": {
    "delivered_flits": 640,
    "drain_ticks": 640,
    "mean_latency_ticks": 241,
    "p95_latency_ticks": 457.0,
    "mean_queue": 240.0,
    "max_queue": 480
  },
  "hotspot_latency_ratio": 241.0
}


## 7. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Balanced mean latency | 1.0000 |
| Hotspot mean latency | 241.0000 |
| Balanced max queue | 0 |
| Hotspot max queue | 480 |
| Latency ratio | 241.000x |


## 8. Explain rather than overclaim

Balanced and hotspot traffic delivered the same 640 flits, but mean latency changed from 1.00 to 241.00 ticks as one output became oversubscribed.

**Evidence boundary:** A transparent mechanism model executed. It establishes the stated relationship under printed assumptions, not native hardware latency, energy, or topology.


## 9. Write the canonical artifact

The next cell stores the environment, metrics, analysis, evidence label, and bounded conclusion, then prints the exact JSON.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 9, "title": 'NoC Routing, Buffers, and Contention', "environment": ENV,
    "evidence_label": 'numerical-model', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Treat congestion as a traffic-placement problem; reduce hotspots or overlap before assuming a faster arithmetic unit will help.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 9,
  "title": "NoC Routing, Buffers, and Contention",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "seed": 20260822
  },
  "evidence_label": "numerical-model",
  "metrics": {
    "balanced": {
      "delivered_flits": 640,
      "drain_ticks": 160,
      "mean_latency_ticks": 1,
      "p95_latency_ticks": 1.0,
      "mean_queue": 0.0,
      "max_queue": 0
    },
    "hotspot": {
      "delivered_flits": 640,
      "drain_ticks": 640,
      "mean_latency_ticks": 241,
      "p95_latency_ticks": 457.0,
      "mean_queue": 240.0,
      "max_queue": 480
    },
    "hotspot_latency_ratio": 241.0
  },
  "analysis": "Balanced and hotspot traffic delivered the same 640 flits, but mean latency changed from 1.00 to 241.00 ticks as one output became oversubscribed.",
  "conclusion": "Treat congestion as a traffic-placement problem; reduce hotspots or over

## 10. Make the decision

> Treat congestion as a traffic-placement problem; reduce hotspots or overlap before assuming a faster arithmetic unit will help.

**Failure analysis:** Real NoCs have multiple hops, routing adaptivity, priorities, virtual channels, credit delays, and topology-specific links. This model proves only the queueing invariant.


## 11. Extend the evidence

Build a mesh model with hop-dependent links, then compare its predictions with Nsight Compute fabric/L2/DRAM stall evidence for a controlled multi-SM kernel.

See [`README.md`](README.md) for the full explanation and references.
